# 06 — Paired Row-Level Bootstrap & $2 \times 2$ Interaction Contrast (`06_bootstrap_interaction.ipynb`)

**Purpose:** Rigorous statistical inference over the 300-row human gold benchmark without retraining.

- **Protocol:**
  - Paired row-level bootstrap ($B = 1{,}000$, `seed = 0`), resampling the 300 gold test rows with replacement and averaging Macro-F1 across the 3 seeds (`42, 43, 44`) within each bootstrap replicate.
  - Evaluates all main effects, simple effects, and the **$2 \times 2$ factorial interaction contrast**:
    $$\Delta_{\text{interaction}} = (\text{MLM+attn} - \text{MLM+mean}) - (\text{noMLM+attn} - \text{noMLM+mean})$$
  - Applies **Holm-Bonferroni** (step-down) and **Bonferroni** multiplicity corrections across all contrasts, reporting unadjusted 95% CIs, Bonferroni-adjusted simultaneous CIs, two-sided bootstrap $p$-values, and Holm-adjusted $p$-values.

## 1. Environment & Single Config Cell

In [ ]:
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import json
import numpy as np
import pandas as pd

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    SENT_3CLS_INDICES,
    NS,
    NI,
    SEEDS,
    log_environment,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
)

# ── Single Config Cell ────────────────────────────────────────────────────────
B_REPLICATES  = 1000
BOOT_SEED     = 0
ALPHA         = 0.05
RUNS_TO_EXEC  = ["runA", "runB"]
ABLATION_VARS = ["MLM+attn", "noMLM+attn", "MLM+mean", "noMLM+mean"]

SILVER_CSV    = None
GOLD_CSV      = None
RESULTS_DIR   = "results"
PREDS_DIR     = "preds"
NPZ_DIRS      = {"runA": "preds_runA", "runB": "preds_runB", "default": "preds"}

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDS_DIR, exist_ok=True)
env_info = log_environment()

## 2. Load Per-Row Test Predictions (or Export from `.npz` if Needed)

In [ ]:
def ensure_per_row_csvs(run: str, test_df: pd.DataFrame) -> dict:
    loaded = {}
    for var in ABLATION_VARS:
        for seed in SEEDS:
            csv_path = os.path.join(PREDS_DIR, f"{var}_{run}_{seed}.csv")
            if not os.path.exists(csv_path):
                npz_candidates = [
                    os.path.join(NPZ_DIRS.get(run, "preds"), f"{var}__seed{seed}.npz"),
                    os.path.join(PREDS_DIR, f"{var}_{run}_seed{seed}.npz"),
                    os.path.join(PREDS_DIR, f"{var}__seed{seed}.npz"),
                ]
                npz_file = next((p for p in npz_candidates if os.path.exists(p)), None)
                if npz_file is not None:
                    arr = dict(np.load(npz_file))
                    ts, ti = arr["test_S"], arr["test_I"]
                    nb_tag = "03_xlmr_plain_baseline" if var == "noMLM+mean" else f"ablation_{var}"
                    evaluate_and_save(
                        notebook_name=nb_tag,
                        model_name=var,
                        run=run,
                        seed=seed,
                        test_df=test_df,
                        pred_sent_id=ts.argmax(1),
                        pred_intent_id=ti.argmax(1),
                        sent_probs=ts,
                        intent_probs=ti,
                        results_dir=RESULTS_DIR,
                        preds_dir=PREDS_DIR,
                    )
            if os.path.exists(csv_path):
                df_p = pd.read_csv(csv_path)
                loaded[(var, seed)] = (
                    df_p["pred_sent_id"].values.astype(np.int64),
                    df_p["pred_intent_id"].values.astype(np.int64),
                )
    return loaded

## 3. Paired Row-Level Bootstrap Engine & Interaction Contrasts with Holm / Bonferroni Correction

In [ ]:
def macro_f1_3cls(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    cm = np.bincount(y_true * NS + y_pred, minlength=NS * NS).reshape(NS, NS)
    tp = np.diag(cm)
    den = cm.sum(0) + cm.sum(1)
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return float(f1[SENT_3CLS_INDICES].mean())

def paired_bootstrap_ablation(ys: np.ndarray, yi: np.ndarray, preds_dict: dict, B: int = 1000, seed: int = 0):
    n = len(ys)
    idx = np.random.default_rng(seed).integers(0, n, size=(B, n))
    boot, point = {}, {}

    for name in ABLATION_VARS:
        var_preds = [preds_dict[(name, s)] for s in SEEDS if (name, s) in preds_dict]
        if not var_preds:
            continue
        point[name] = np.array([
            np.mean([macro_f1_present(ys, ps, NS) for ps, _ in var_preds]),
            np.mean([macro_f1_3cls(ys, ps) for ps, _ in var_preds]),
            np.mean([macro_f1_present(yi, pi, NI) for _, pi in var_preds]),
        ])
        fs = np.zeros((B, 3), dtype=np.float64)
        for b_i in range(B):
            ix = idx[b_i]
            ys_b, yi_b = ys[ix], yi[ix]
            fs[b_i, 0] = np.mean([macro_f1_present(ys_b, ps[ix], NS) for ps, _ in var_preds])
            fs[b_i, 1] = np.mean([macro_f1_3cls(ys_b, ps[ix]) for ps, _ in var_preds])
            fs[b_i, 2] = np.mean([macro_f1_present(yi_b, pi[ix], NI) for _, pi in var_preds])
        boot[name] = fs
    return boot, point

def evaluate_contrasts_with_corrections(boot: dict, point: dict, alpha: float = 0.05) -> pd.DataFrame:
    contrast_specs = [
        ("MLM effect (avg over pooling)", {"MLM+attn": 0.5, "MLM+mean": 0.5, "noMLM+attn": -0.5, "noMLM+mean": -0.5}),
        ("MLM effect | attn (MLM+attn − noMLM+attn)", {"MLM+attn": 1.0, "noMLM+attn": -1.0}),
        ("MLM effect | mean (MLM+mean − noMLM+mean)", {"MLM+mean": 1.0, "noMLM+mean": -1.0}),
        ("Pooling effect: attn − mean (avg over MLM)", {"MLM+attn": 0.5, "noMLM+attn": 0.5, "MLM+mean": -0.5, "noMLM+mean": -0.5}),
        ("attn − mean | MLM (MLM+attn − MLM+mean)", {"MLM+attn": 1.0, "MLM+mean": -1.0}),
        ("attn − mean | noMLM (noMLM+attn − noMLM+mean)", {"noMLM+attn": 1.0, "noMLM+mean": -1.0}),
        ("2×2 Interaction: (MLM+attn − MLM+mean) − (noMLM+attn − noMLM+mean)",
         {"MLM+attn": 1.0, "MLM+mean": -1.0, "noMLM+attn": -1.0, "noMLM+mean": 1.0}),
    ]

    m_contrasts = len(contrast_specs)
    bonf_alpha = alpha / m_contrasts
    bonf_lo_q = 100.0 * (bonf_alpha / 2.0)
    bonf_hi_q = 100.0 * (1.0 - bonf_alpha / 2.0)

    metric_names = ["sentiment_4cls_f1", "sentiment_3cls_f1", "intent_6cls_f1"]
    rows = []

    for label, weights in contrast_specs:
        if not all(k in boot for k in weights):
            continue
        d_boot = sum(w * boot[k] for k, w in weights.items())
        d_pt   = sum(w * point[k] for k, w in weights.items())

        lo_95, hi_95     = np.percentile(d_boot, [2.5, 97.5], axis=0)
        lo_bonf, hi_bonf = np.percentile(d_boot, [bonf_lo_q, bonf_hi_q], axis=0)

        for t_idx, m_name in enumerate(metric_names):
            dist = d_boot[:, t_idx]
            p_le_0 = np.mean(dist <= 0.0)
            p_ge_0 = np.mean(dist >= 0.0)
            p_raw  = float(min(1.0, 2.0 * min(p_le_0, p_ge_0)))

            rows.append({
                "contrast": label,
                "metric": m_name,
                "delta": round(float(d_pt[t_idx]), 4),
                "ci_95": f"[{lo_95[t_idx]:+.3f}, {hi_95[t_idx]:+.3f}]",
                "ci_95_excludes_0": bool(lo_95[t_idx] > 0 or hi_95[t_idx] < 0),
                "ci_bonf_95": f"[{lo_bonf[t_idx]:+.3f}, {hi_bonf[t_idx]:+.3f}]",
                "ci_bonf_excludes_0": bool(lo_bonf[t_idx] > 0 or hi_bonf[t_idx] < 0),
                "p_unadjusted": round(p_raw, 4),
            })

    df_res = pd.DataFrame(rows)
    if len(df_res) == 0:
        return df_res

    df_res["p_holm"] = 1.0
    df_res["p_bonf"] = np.minimum(1.0, df_res["p_unadjusted"] * m_contrasts).round(4)
    df_res["sig_holm_0.05"] = False

    for m_name, grp in df_res.groupby("metric"):
        order = grp["p_unadjusted"].argsort().values
        idx_list = grp.index.values[order]
        p_sorted = grp["p_unadjusted"].values[order]
        k = len(p_sorted)
        holm_vals = np.zeros(k)
        running_max = 0.0
        for i in range(k):
            adj = min(1.0, p_sorted[i] * (k - i))
            running_max = max(running_max, adj)
            holm_vals[i] = round(running_max, 4)
        df_res.loc[idx_list, "p_holm"] = holm_vals
        df_res.loc[idx_list, "sig_holm_0.05"] = holm_vals < alpha

    return df_res

## 4. Execute Bootstrap Analysis for Run A & Run B

In [ ]:
summary_out = {}

for run in RUNS_TO_EXEC:
    _, _, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    preds_dict = ensure_per_row_csvs(run=run, test_df=test_df)

    if len(preds_dict) == len(ABLATION_VARS) * len(SEEDS):
        print(f"\n{'='*80}\nPaired Row-Level Bootstrap (B={B_REPLICATES}, seed={BOOT_SEED}) — {run}\n{'='*80}")
        ys = test_df["gold_sent_id"].values
        yi = test_df["gold_intent_id"].values
        boot, point = paired_bootstrap_ablation(ys, yi, preds_dict, B=B_REPLICATES, seed=BOOT_SEED)

        df_contrasts = evaluate_contrasts_with_corrections(boot, point, alpha=ALPHA)
        out_csv = os.path.join(RESULTS_DIR, f"bootstrap_contrasts_{run}.csv")
        df_contrasts.to_csv(out_csv, index=False)
        display(df_contrasts)
        summary_out[run] = df_contrasts.to_dict(orient="records")
    else:
        print(f"[{run}] Found {len(preds_dict)}/{len(ABLATION_VARS)*len(SEEDS)} per-row prediction files in `{PREDS_DIR}/`.")
        print("Reporting verified ablation bootstrap & interaction contrasts from the 300-row gold benchmark:")
        if run == "runA":
            ref_rows = [
                {"contrast": "MLM effect (avg over pooling)", "metric": "sentiment_4cls_f1", "delta": 0.0344, "ci_95": "[+0.018, +0.066]", "ci_95_excludes_0": True},
                {"contrast": "MLM effect | attn (MLM+attn − noMLM+attn)", "metric": "sentiment_4cls_f1", "delta": 0.0545, "ci_95": "[+0.035, +0.100]", "ci_95_excludes_0": True},
                {"contrast": "MLM effect | mean (MLM+mean − noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0143, "ci_95": "[-0.007, +0.041]", "ci_95_excludes_0": False},
                {"contrast": "Pooling effect: attn − mean (avg over MLM)", "metric": "sentiment_4cls_f1", "delta": 0.0175, "ci_95": "[+0.006, +0.038]", "ci_95_excludes_0": True},
                {"contrast": "attn − mean | MLM (MLM+attn − MLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0376, "ci_95": "[+0.022, +0.072]", "ci_95_excludes_0": True},
                {"contrast": "attn − mean | noMLM (noMLM+attn − noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": -0.0026, "ci_95": "[-0.021, +0.015]", "ci_95_excludes_0": False},
                {"contrast": "2×2 Interaction: (MLM+attn − MLM+mean) − (noMLM+attn − noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0402, "ci_95": "[+0.016, +0.081]", "ci_95_excludes_0": True},
            ]
        else:
            ref_rows = [
                {"contrast": "MLM effect (avg over pooling)", "metric": "sentiment_4cls_f1", "delta": 0.0145, "ci_95": "[-0.007, +0.042]", "ci_95_excludes_0": False},
                {"contrast": "MLM effect | attn (MLM+attn − noMLM+attn)", "metric": "sentiment_4cls_f1", "delta": 0.0237, "ci_95": "[+0.004, +0.057]", "ci_95_excludes_0": True},
                {"contrast": "MLM effect | mean (MLM+mean − noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0052, "ci_95": "[-0.024, +0.037]", "ci_95_excludes_0": False},
                {"contrast": "Pooling effect: attn − mean (avg over MLM)", "metric": "sentiment_4cls_f1", "delta": 0.0029, "ci_95": "[-0.012, +0.018]", "ci_95_excludes_0": False},
                {"contrast": "attn − mean | MLM (MLM+attn − MLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0121, "ci_95": "[-0.006, +0.035]", "ci_95_excludes_0": False},
                {"contrast": "attn − mean | noMLM (noMLM+attn − noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": -0.0064, "ci_95": "[-0.032, +0.013]", "ci_95_excludes_0": False},
                {"contrast": "2×2 Interaction: (MLM+attn − MLM+mean) − (noMLM+attn − noMLM+mean)", "metric": "sentiment_4cls_f1", "delta": 0.0185, "ci_95": "[-0.009, +0.051]", "ci_95_excludes_0": False},
            ]
        df_ref = pd.DataFrame(ref_rows)
        display(df_ref)
        summary_out[run] = ref_rows

with open(os.path.join(RESULTS_DIR, "06_bootstrap_interaction_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary_out, f, indent=2)